In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv('retail_store_sales.csv')

print(df.shape)
df.head()

(12575, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [18]:
df = pd.read_csv('retail_store_sales.csv')

In [19]:
print(df.shape)
df.head()

(12575, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [20]:
# Data Quality Report

quality_report = pd.DataFrame({
    'Data Type': df.dtypes,
    'Null Count': df.isnull().sum(),
    'Unique Values': df.nunique()
})

display(quality_report)

print("Duplicate Rows:", df.duplicated().sum())

print("\nNumeric Value Ranges:")
display(df.select_dtypes(include=np.number).agg(['min', 'max']).T)

,Data Type,Null Count,Unique Values
Transaction ID,object,0,12575
Customer ID,object,0,25
Category,object,0,8
Item,object,1213,200
Price Per Unit,float64,609,25
Quantity,float64,604,10
Total Spent,float64,604,227
Payment Method,object,0,3
Location,object,0,2
Transaction Date,object,0,1114


Duplicate Rows: 0

Numeric Value Ranges:


,min,max
Price Per Unit,5.0,41.0
Quantity,1.0,10.0
Total Spent,5.0,410.0


In [21]:
for col in ['Category', 'Payment Method', 'Location', 'Discount Applied']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))


Category:
Category
Electric household essentials         1591
Furniture                             1591
Food                                  1588
Milk Products                         1584
Butchers                              1568
Beverages                             1567
Computers and electric accessories    1558
Patisserie                            1528
Name: count, dtype: int64

Payment Method:
Payment Method
Cash              4310
Digital Wallet    4144
Credit Card       4121
Name: count, dtype: int64

Location:
Location
Online      6354
In-store    6221
Name: count, dtype: int64

Discount Applied:
Discount Applied
True     4219
NaN      4199
False    4157
Name: count, dtype: int64


In [22]:
before_rows = len(df)
before_nulls = df.isnull().sum().sum()
before_duplicates = df.duplicated().sum()
before_dtypes = df.dtypes.astype(str)

In [23]:
# 1. Fill missing Price Per Unit using Total Spent / Quantity
mask = df['Price Per Unit'].isna()

df.loc[mask, 'Price Per Unit'] = (
    df.loc[mask, 'Total Spent'] / df.loc[mask, 'Quantity']
)

# 2. Recover missing Item using Category + Price Per Unit
item_map = (
    df.dropna(subset=['Item', 'Price Per Unit'])
      .drop_duplicates(['Category', 'Price Per Unit'])
      .set_index(['Category', 'Price Per Unit'])['Item']
)

def fill_item(row):
    if pd.isna(row['Item']):
        return item_map.get(
            (row['Category'], row['Price Per Unit']),
            row['Item']
        )
    return row['Item']

df['Item'] = df.apply(fill_item, axis=1)

# 3. Fill missing Quantity using the median
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())

# 4. Fill missing Total Spent using Price × Quantity
df['Total Spent'] = df['Total Spent'].fillna(
    df['Price Per Unit'] * df['Quantity']
)

# 5. Fill missing Discount Applied using the mode
df['Discount Applied'] = df['Discount Applied'].fillna(
    df['Discount Applied'].mode()[0]
)

# Check remaining missing values
df.isnull().sum()

C:\Users\nguru\AppData\Local\Temp\ipykernel_19916\4294530300.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Discount Applied'] = df['Discount Applied'].fillna(


Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64

### Missing Data Handling

- Missing `Price Per Unit` values were calculated from `Total Spent / Quantity` because these columns have a direct mathematical relationship.
- Missing `Item` values were recovered using the matching `Category` and `Price Per Unit`.
- Missing `Quantity` values were filled with the median because the median is less sensitive to extreme values.
- Missing `Total Spent` values were calculated using `Price Per Unit × Quantity`.
- Missing `Discount Applied` values were filled using the mode because it is a categorical variable.

In [24]:
# Check duplicates before removal
duplicates_before = df.duplicated().sum()
print("Duplicate rows before removal:", duplicates_before)

# Remove duplicate rows
df = df.drop_duplicates()

duplicates_after = df.duplicated().sum()
print("Duplicate rows after removal:", duplicates_after)

print("Duplicates removed:", duplicates_before - duplicates_after)

Duplicate rows before removal: 0
Duplicate rows after removal: 0
Duplicates removed: 0


### Duplicate Removal

The dataset was checked for duplicate rows. No duplicate rows were found, so no records were removed.

In [25]:
# Standardisation

# Remove unnecessary spaces from text columns
text_columns = ['Category', 'Item', 'Payment Method', 'Location']

for col in text_columns:
    df[col] = df[col].astype('string').str.strip()

# Convert Transaction Date to proper datetime format
df['Transaction Date'] = pd.to_datetime(
    df['Transaction Date'],
    errors='coerce'
)

# Standardise Discount Applied as boolean
df['Discount Applied'] = df['Discount Applied'].astype('boolean')

# Check results
print(df.dtypes)
print("\nInvalid dates after conversion:", df['Transaction Date'].isnull().sum())

Transaction ID              object
Customer ID                 object
Category            string[python]
Item                string[python]
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method      string[python]
Location            string[python]
Transaction Date    datetime64[ns]
Discount Applied           boolean
dtype: object

Invalid dates after conversion: 0


### Standardisation

Text columns were checked and unnecessary whitespace was removed to ensure consistent formatting. The `Transaction Date` column was converted to a standard datetime format, while `Discount Applied` was standardised as a boolean field.

In [26]:
# Outlier Detection using IQR

numeric_columns = ['Price Per Unit', 'Quantity', 'Total Spent']

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f"{col}:")
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Number of Outliers:", len(outliers))
    print()

Price Per Unit:
Lower Bound: -15.25
Upper Bound: 62.75
Number of Outliers: 0

Quantity:
Lower Bound: -4.5
Upper Bound: 15.5
Number of Outliers: 0

Total Spent:
Lower Bound: -158.0
Upper Bound: 402.0
Number of Outliers: 60



### Outlier Detection

The IQR method was used to detect outliers in the numerical columns.

- `Price Per Unit` had 0 outliers.
- `Quantity` had 0 outliers.
- `Total Spent` had 60 outliers.

The 60 outliers in `Total Spent` were retained because they represent valid high-value transactions rather than obvious data-entry errors.

In [27]:
# Data Type Correction

df['Transaction ID'] = df['Transaction ID'].astype('string')
df['Customer ID'] = df['Customer ID'].astype('string')

df['Price Per Unit'] = df['Price Per Unit'].astype(float)
df['Quantity'] = df['Quantity'].astype(float)
df['Total Spent'] = df['Total Spent'].astype(float)

df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])

df['Discount Applied'] = df['Discount Applied'].astype('boolean')

# Check final data types
df.dtypes

Transaction ID      string[python]
Customer ID         string[python]
Category            string[python]
Item                string[python]
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method      string[python]
Location            string[python]
Transaction Date    datetime64[ns]
Discount Applied           boolean
dtype: object

### Data Type Correction

Data types were corrected to make the dataset analysis-ready. Transaction and Customer IDs were stored as strings, monetary fields were converted to floats, Transaction Date was stored as datetime, and Discount Applied was stored as a boolean.

In [28]:
# Save after-cleaning metrics
after_rows = len(df)
after_nulls = df.isnull().sum().sum()
after_duplicates = df.duplicated().sum()

# Count columns with correct final data types
expected_types = {
    'Transaction ID': 'string',
    'Customer ID': 'string',
    'Category': 'string',
    'Item': 'string',
    'Price Per Unit': 'float',
    'Quantity': 'float',
    'Total Spent': 'float',
    'Payment Method': 'string',
    'Location': 'string',
    'Transaction Date': 'datetime',
    'Discount Applied': 'boolean'
}

def dtype_accuracy(dtypes):
    correct = 0

    for col, expected in expected_types.items():
        dtype = str(dtypes[col])

        if expected == 'string' and ('string' in dtype or dtype == 'object'):
            correct += 1
        elif expected == 'float' and 'float' in dtype:
            correct += 1
        elif expected == 'datetime' and 'datetime' in dtype:
            correct += 1
        elif expected == 'boolean' and 'bool' in dtype:
            correct += 1

    return f"{correct}/{len(expected_types)}"

before_accuracy = dtype_accuracy(before_dtypes)
after_accuracy = dtype_accuracy(df.dtypes)

summary = pd.DataFrame({
    'Metric': [
        'Row Count',
        'Null Values',
        'Duplicate Rows',
        'Correct Data Types'
    ],
    'Before Cleaning': [
        before_rows,
        before_nulls,
        before_duplicates,
        before_accuracy
    ],
    'After Cleaning': [
        after_rows,
        after_nulls,
        after_duplicates,
        after_accuracy
    ]
})

summary

,Metric,Before Cleaning,After Cleaning
0,Row Count,12575,12575
1,Null Values,7229,0
2,Duplicate Rows,0,0
3,Correct Data Types,9/11,11/11


In [29]:
df.to_csv('retail_store_sales_cleaned.csv', index=False)
print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
